In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/silver_utils

In [0]:
# Cell 1: Setup, Context & Environment Initialization
db_helpers = DatabricksHelpers(dbutils)
current_user_email = db_helpers.current_user()

catalog_name = "f1_dev"
schemas = ["bronze", "silver", "gold"]

# Ensure bronze, silver, and gold schemas exist in Unity Catalog
initialize_environment(spark, catalog_name, schemas)

source_table = f"{catalog_name}.bronze.constructors"
target_table = f"{catalog_name}.silver.dim_constructors"

# --- CELL 1 TEST ---
assert current_user_email and ("@" in current_user_email or current_user_email == "root"), \
    f"TEST FAILED: Invalid user context derived: '{current_user_email}'"

assert spark.catalog.tableExists(source_table), \
    f"TEST FAILED: Source bronze table '{source_table}' does not exist in catalog."

print(f"✓ Cell 1 Passed: Executed by '{current_user_email}' on catalog '{catalog_name}'.")

In [0]:
# Cell 2: Data Extraction & Bronze String Sanitization
df_bronze = spark.table(source_table)

# Reusable string cleaning: purges '\N', 'NULL', 'null', and empty strings across all text fields
df_sanitized = sanitize_bronze_strings(df_bronze)

# --- CELL 2 TEST ---
string_cols = [f.name for f in df_sanitized.schema.fields if f.dataType.typeName() == "string"]
for col_name in string_cols:
    dirty_count = df_sanitized.filter(
        (F.col(col_name) == r"\N") | (F.lower(F.col(col_name)) == "null")
    ).count()
    assert dirty_count == 0, f"TEST FAILED: Found {dirty_count} unhandled string literals in '{col_name}'"

print(f"✓ Cell 2 Passed: String sanitization verified across {len(string_cols)} string columns.")

In [0]:
# Cell 3: Silver Domain Transformations & Safe Casting
df_constructors_silver = (
    df_sanitized
    .select(
        F.col("constructorId").try_cast("integer").alias("constructor_id"),
        F.col("constructorRef").alias("constructor_ref"),
        F.col("name").alias("constructor_name"),
        F.col("nationality")
    )
    # Deduplicate against primary key
    .dropDuplicates(["constructor_id"])
)

# --- CELL 3 TEST ---
sample_row = df_constructors_silver.first()

# 1. Primary key data type verification
assert dict(df_constructors_silver.dtypes)["constructor_id"] == "int", \
    "TEST FAILED: constructor_id was not successfully cast to integer."

# 2. Confirm unneeded/raw columns (like raw url if present) were excluded
prohibited_cols = {"url"}
assert not prohibited_cols.intersection(set(df_constructors_silver.columns)), \
    f"TEST FAILED: Unneeded columns detected in Silver projection: {prohibited_cols.intersection(set(df_constructors_silver.columns))}"

print("✓ Cell 3 Passed: Constructor transformations and column projections verified.")

In [0]:
# Cell 4: Pre-Write Data Quality Circuit Breaker
# Reusable Circuit Breaker: Enforces PK Non-Null, Key Uniqueness, and PII Exclusion
run_circuit_breaker(
    df=df_constructors_silver, 
    primary_keys=["constructor_id"], 
    prohibited_pii={"url"}
)

print("✓ Cell 4 Passed: Pre-write quality circuit breaker executed successfully.")

In [0]:
# Cell 5: Target Delta DDL Creation & Idempotent Delta Load
# 1. Ensure target Silver Delta table exists with Primary Key constraint
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        constructor_id INT NOT NULL,
        constructor_ref STRING,
        constructor_name STRING,
        nationality STRING,
        CONSTRAINT dim_constructors_pk PRIMARY KEY (constructor_id)
    )
    USING DELTA
""")

# 2. Reusable Upsert: Enables CDF, Auto-Optimize, and executes SCD Type 1 MERGE
upsert_silver_delta(
    spark=spark, 
    df=df_constructors_silver, 
    target_table=target_table, 
    primary_keys=["constructor_id"]
)

# --- CELL 5 TEST ---
target_df = spark.table(target_table)
staging_count = df_constructors_silver.count()
target_count = target_df.count()

# Confirm written count matches staging expectations
assert target_count >= staging_count, \
    f"TEST FAILED: Target table count ({target_count}) is lower than staging count ({staging_count})."

print(f"✓ Cell 5 Passed: Delta MERGE load to '{target_table}' completed and verified.")

In [0]:
# Cell 6: Unity Catalog Metadata Governance & Exit
# Reusable Governance Tagger: Sets tags and asserts existence in information_schema
apply_silver_tags(
    spark=spark, 
    active_catalog=catalog_name, 
    target_table=target_table, 
    current_user_email=current_user_email, 
    table_type="dimension"
)

# Clean exit for Databricks Workflows / Jobs Orchestration
dbutils.notebook.exit(f"SUCCESS: Pipeline executed cleanly by {current_user_email} on {target_table}")